# OldCodeDon'tStudy 

In [1]:

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

# Load features
image_features = np.load("/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/coco_features/image_features.npy")
caption_features = np.load("/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/coco_features/caption_features.npy")
labels = np.load("/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/coco_features/labels.npy")

# Early fusion
X = np.concatenate([image_features, caption_features], axis=1)  # (5000, 2816)
y = labels  # (5000, 80)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# MLP model
class MLP(nn.Module):
    def __init__(self, input_dim=2816, output_dim=80):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim),
            nn.Sigmoid()  # multilabel
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
epochs = 100
batch_size = 64

for epoch in range(epochs):
    model.train()
    perm = torch.randperm(X_train.size(0))
    total_loss = 0
    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]
        xb, yb = X_train[idx].to(device), y_train[idx].to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    preds = model(X_test.to(device))
    test_loss = criterion(preds, y_test.to(device)).item()
print("Final Test Loss:", test_loss)


X shape: (4952, 2816)
y shape: (4952, 80)
Epoch 1/100, Loss: 10.1632
Epoch 2/100, Loss: 7.7016
Epoch 3/100, Loss: 7.1490
Epoch 4/100, Loss: 6.3399
Epoch 5/100, Loss: 5.6914
Epoch 6/100, Loss: 5.2104
Epoch 7/100, Loss: 4.9399
Epoch 8/100, Loss: 4.8294
Epoch 9/100, Loss: 4.6328
Epoch 10/100, Loss: 4.4816
Epoch 11/100, Loss: 4.3305
Epoch 12/100, Loss: 4.2457
Epoch 13/100, Loss: 4.1053
Epoch 14/100, Loss: 4.0835
Epoch 15/100, Loss: 3.8721
Epoch 16/100, Loss: 3.7778
Epoch 17/100, Loss: 3.6344
Epoch 18/100, Loss: 3.5306
Epoch 19/100, Loss: 3.4545
Epoch 20/100, Loss: 3.3682
Epoch 21/100, Loss: 3.2611
Epoch 22/100, Loss: 3.1191
Epoch 23/100, Loss: 3.0403
Epoch 24/100, Loss: 3.0007
Epoch 25/100, Loss: 2.8523
Epoch 26/100, Loss: 2.8505
Epoch 27/100, Loss: 2.6912
Epoch 28/100, Loss: 2.6333
Epoch 29/100, Loss: 2.5186
Epoch 30/100, Loss: 2.4824
Epoch 31/100, Loss: 2.4365
Epoch 32/100, Loss: 2.3797
Epoch 33/100, Loss: 2.2965
Epoch 34/100, Loss: 2.2103
Epoch 35/100, Loss: 2.1050
Epoch 36/100, Loss: 2

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model.eval()
with torch.no_grad():
    preds = model(X_test.to(device))
    test_loss = criterion(preds, y_test.to(device)).item()
    acc = accuracy_score(y_test, preds.cpu().numpy().round())
print("Final Test Loss:", test_loss)
print("Final Test Accuracy:", acc)

Final Test Loss: 0.23584575951099396
Final Test Accuracy: 0.2391523713420787


# NewCodeStudy

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import TensorDataset, DataLoader

# -------------------------
# Load your arrays (same as you had)
image_features = np.load("/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/coco_features/image_features.npy")
caption_features = np.load("/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/coco_features/caption_features.npy")
labels = np.load("/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/coco_features/labels.npy")

X = np.concatenate([image_features, caption_features], axis=1)  # (5000, 2816)
y = labels  # (5000, 80)

# Train / val / test split (keep test for final eval)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.2, random_state=42)  # val is 16% overall

# Standardize features (fit on train only)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Convert to tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val, dtype=torch.float32)
y_val_t   = torch.tensor(y_val, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32)

# DataLoaders
batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=512, shuffle=False)

# -------------------------
# Model (NO Sigmoid at the end -> outputs logits)
class MLP(nn.Module):
    def __init__(self, input_dim=2816, output_dim=80, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(256, output_dim)   # logits
        )
    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=X.shape[1], output_dim=y.shape[1], drop=0.3).to(device)

# -------------------------
# Loss with pos_weight to handle class imbalance
# compute pos_weight = (#negatives / #positives) per class from train
pos_counts = y_train.sum(axis=0) + 1e-6
neg_counts = (y_train.shape[0] - y_train.sum(axis=0)) + 1e-6
pos_weight = torch.tensor(neg_counts / pos_counts, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# -------------------------
# Training with validation, early stopping
epochs = 50
best_val = -1.0
patience = 6
bad = 0
best_state = None

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # optional clipping
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    avg_train_loss = running_loss / len(train_loader.dataset)

    # validate using sigmoid -> threshold 0.5 for quick check
    model.eval()
    with torch.no_grad():
        v_logits = model(X_val_t.to(device))
        v_probs = torch.sigmoid(v_logits).cpu().numpy()
        v_pred = (v_probs >= 0.5).astype(int)
        v_true = y_val_t.numpy()
        v_micro_f1 = f1_score(v_true, v_pred, average='micro', zero_division=0)

    print(f"Epoch {epoch+1}/{epochs} | Train loss: {avg_train_loss:.4f} | Val micro-F1: {v_micro_f1:.4f}")

    # scheduler on metric
    scheduler.step(v_micro_f1)

    # early stopping
    if v_micro_f1 > best_val:
        best_val = v_micro_f1
        bad = 0
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= patience:
            print("Early stopping.")
            break

# load best model
if best_state is not None:
    model.load_state_dict(best_state)
model.to(device)

# -------------------------
# Tune per-class thresholds on validation set (coarse search)
model.eval()
with torch.no_grad():
    val_logits = model(X_val_t.to(device)).cpu().numpy()
val_probs = 1 / (1 + np.exp(-val_logits))  # sigmoid
y_val_np = y_val_t.numpy()

num_classes = y_val_np.shape[1]
best_thresh = np.full(num_classes, 0.5)
for c in range(num_classes):
    best_f1 = 0.0
    # coarse grid
    for t in np.linspace(0.05, 0.95, 19):
        f1 = f1_score(y_val_np[:, c], (val_probs[:, c] >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh[c] = t

# -------------------------
# Final evaluation on test set using tuned thresholds
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t.to(device)).cpu().numpy()
test_probs = 1 / (1 + np.exp(-test_logits))
y_test_np = y_test_t.numpy()
y_pred_opt = (test_probs >= best_thresh).astype(int)

print("Final Test Metrics (with tuned thresholds):")
print("Micro F1:", f1_score(y_test_np, y_pred_opt, average='micro', zero_division=0))
print("Macro F1:", f1_score(y_test_np, y_pred_opt, average='macro', zero_division=0))
print("Micro Precision:", precision_score(y_test_np, y_pred_opt, average='micro', zero_division=0))
print("Micro Recall:", recall_score(y_test_np, y_pred_opt, average='micro', zero_division=0))


Epoch 1/50 | Train loss: 0.8146 | Val micro-F1: 0.3084
Epoch 2/50 | Train loss: 0.4885 | Val micro-F1: 0.3520
Epoch 3/50 | Train loss: 0.3990 | Val micro-F1: 0.3913
Epoch 4/50 | Train loss: 0.3519 | Val micro-F1: 0.4076
Epoch 5/50 | Train loss: 0.3209 | Val micro-F1: 0.4096
Epoch 6/50 | Train loss: 0.2929 | Val micro-F1: 0.4348
Epoch 7/50 | Train loss: 0.2779 | Val micro-F1: 0.4253
Epoch 8/50 | Train loss: 0.2612 | Val micro-F1: 0.4528
Epoch 9/50 | Train loss: 0.2495 | Val micro-F1: 0.4651
Epoch 10/50 | Train loss: 0.2317 | Val micro-F1: 0.4695
Epoch 11/50 | Train loss: 0.2244 | Val micro-F1: 0.4757
Epoch 12/50 | Train loss: 0.2237 | Val micro-F1: 0.4784
Epoch 13/50 | Train loss: 0.2127 | Val micro-F1: 0.4930
Epoch 14/50 | Train loss: 0.1996 | Val micro-F1: 0.4956
Epoch 15/50 | Train loss: 0.1964 | Val micro-F1: 0.5015
Epoch 16/50 | Train loss: 0.1910 | Val micro-F1: 0.5021
Epoch 17/50 | Train loss: 0.1851 | Val micro-F1: 0.5072
Epoch 18/50 | Train loss: 0.1837 | Val micro-F1: 0.5080
E

/tmp/ipykernel_7301/1012587773.py:152: RuntimeWarning: overflow encountered in exp
  test_probs = 1 / (1 + np.exp(-test_logits))
